# Upload model I/O as a dataset

Save model inputs and outputs as JSONL, then upload the file to Burstchester as a dataset.

In [ ]:
# 1. Clone the CLI repository.
!git clone https://github.com/tomongoose/burstchester.git /content/burstchester
%cd /content/burstchester

In [ ]:
# 2. Load Burstchester access token from Colab secrets first.
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

token = os.environ.get('BURSTCHESTER_ACCESS_TOKEN')
if not token and userdata is not None:
    try:
        token = userdata.get('BURSTCHESTER_ACCESS_TOKEN')
    except Exception:
        token = None
if not token:
    token = getpass('Burstchester access token: ')
if not token.strip():
    raise ValueError('BURSTCHESTER_ACCESS_TOKEN is required.')
os.environ['BURSTCHESTER_ACCESS_TOKEN'] = token.strip()
print('Burstchester token configured.')

In [ ]:
# 3. Save model input/output pairs as JSONL.
import json
from pathlib import Path

samples = [
    {
        'messages': [
            {'role': 'user', 'content': 'Write a short product tagline for an AI dataset platform.'},
            {'role': 'assistant', 'content': 'Collect, improve, and share trusted training data for better AI models.'},
        ]
    },
    {
        'messages': [
            {'role': 'user', 'content': 'Explain why dataset quality matters when fine-tuning a Gemma model.'},
            {'role': 'assistant', 'content': 'Fine-tuning data directly shapes the response patterns and knowledge boundaries the model learns, so accurate and consistent input/output pairs are essential.'},
        ]
    },
]

dataset_path = Path('/content/model-io-dataset.jsonl')
with dataset_path.open('w', encoding='utf-8') as f:
    for sample in samples:
        f.write(json.dumps(sample, ensure_ascii=False) + '\n')

print(dataset_path)

In [ ]:
# 4. Upload the JSONL file as a Burstchester dataset.
!node cli/src/cli.mjs upload-dataset \
  --file /content/model-io-dataset.jsonl \
  --title 'Gemma model IO sample' \
  --description 'Model input/output pairs collected for supervised fine-tuning.' \
  --tags 'model-io,sft,gemma' \
  --source-model 'google/gemma-4-E2B' \
  --base-model-hint 'google/gemma-4-E2B' \
  --task-type 'chat' \
  --language 'en' \
  --license 'cc-by-4.0' \
  --point-cost '10'

To capture real model requests and responses through a proxy, run `proxy-record` in a separate terminal and upload the captured file with `upload-proxy-log`. Colab is not ideal for keeping a local proxy running for a long time, so this notebook provides the direct JSONL authoring flow as the runnable example.